# 02. Tabular Baselines

This notebook treats the task as wallet-level supervised classification and ignores the edge list.

Recommended default:

- feature block: `eth_twitter_combined_features_*`
- target: `label`
- main metric: `PR-AUC`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
nodes_df = pd.read_csv(ROOT / "wash_trading_gnn_nodes_10000.csv")


In [ ]:
FEATURE_GROUP = "eth_twitter_combined_features"
RANDOM_STATE = 42

feature_groups = {
    "features": [c for c in nodes_df.columns if c.startswith("features_")],
    "normalized_log_features": [c for c in nodes_df.columns if c.startswith("normalized_log_features_")],
    "twitter_combined_features": [c for c in nodes_df.columns if c.startswith("twitter_combined_features_")],
    "eth_twitter_combined_features": [c for c in nodes_df.columns if c.startswith("eth_twitter_combined_features_")],
}

feature_cols = feature_groups[FEATURE_GROUP]
X = nodes_df[feature_cols].copy()
y = nodes_df["label"].copy()

train_idx, temp_idx = train_test_split(
    nodes_df.index,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y.loc[temp_idx],
)

X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_val, y_val = X.loc[val_idx], y.loc[val_idx]
X_test, y_test = X.loc[test_idx], y.loc[test_idx]

print(X_train.shape, X_val.shape, X_test.shape)


In [ ]:
def best_threshold(y_true, y_prob, thresholds=None):
    thresholds = np.linspace(0.05, 0.95, 37) if thresholds is None else thresholds
    best = {"threshold": 0.5, "f1": -1.0}
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(threshold), "f1": float(f1)}
    return best["threshold"]


def evaluate_binary_classifier(model_name, y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "model": model_name,
        "threshold": threshold,
        "PR-AUC": average_precision_score(y_true, y_prob),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Balanced-Accuracy": balanced_accuracy_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
    }


In [ ]:
scale_pos_weight = y_train.eq(0).sum() / y_train.eq(1).sum()

models = {
    "logreg": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
        ]
    ),
    "random_forest": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "mlp": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                MLPClassifier(
                    hidden_layer_sizes=(64, 32),
                    activation="relu",
                    learning_rate_init=1e-3,
                    max_iter=200,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "xgboost": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                XGBClassifier(
                    n_estimators=300,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    reg_lambda=1.0,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    scale_pos_weight=scale_pos_weight,
                    tree_method="hist",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}


In [ ]:
results = []
fitted_models = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    val_prob = model.predict_proba(X_val)[:, 1]
    threshold = best_threshold(y_val.to_numpy(), val_prob)
    test_prob = model.predict_proba(X_test)[:, 1]
    results.append(evaluate_binary_classifier(model_name, y_test.to_numpy(), test_prob, threshold))
    fitted_models[model_name] = model

results_df = pd.DataFrame(results).sort_values(["PR-AUC", "F1"], ascending=False)
display(results_df)


In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]
best_model


## Notes

- Keep `PR-AUC` as the primary leaderboard metric.
- `XGBoost` is usually the strongest baseline here.
- The next notebook adds graph-derived features to test whether message passing is actually necessary.
